In [1]:
import pandas as pd
import numpy as np

prices = [
    100, 102, 104, 101, 98,
    105, 110, 108, 112, 115,
    109, 107, 111, 117, 120,
    118, 121, 125, 123, 128,
    130, 127, 132, 135, 138,
    136, 140, 145, 143, 148,
    150, 147, 152, 155, 153,
    158, 160, 157, 162, 165
]

best_fast = 5
best_slow = 6
results = []

df = pd.DataFrame({'Prices' : prices})
df['Return'] = df['Prices'].pct_change()
df['Slow_MA'] = df['Prices'].rolling(window = best_slow).mean()
df['Fast_MA'] = df['Prices'].rolling(window = best_fast).mean()
df['Signal'] = np.where(df['Fast_MA'] >df['Slow_MA'],1,0 )
df['Strategy_Return'] = df['Signal'].shift(1) * df['Return']
df['Equity_Curve'] = (1+ df['Strategy_Return']).cumprod()
sharpe = df['Strategy_Return'].mean()/df['Strategy_Return'].std()
final_equity = df['Equity_Curve'].iloc[-1]
total_return = final_equity -1 
drawdown = (df['Equity_Curve'] - df['Equity_Curve'].cummax() )/df['Equity_Curve'].cummax()
max_drawdown  = drawdown.min()

df['Position'] = df['Signal'].diff()
trade_entries = df[df['Position'] == 1]['Prices']
trade_exits = df[df['Position'] == -1]['Prices']
if(len(trade_entries) > len(trade_exits)):
    trade_entries = trade_entries[:-1]

trade_returns = (trade_exits.values - trade_entries.values)/trade_entries.values
win_rate = (trade_returns > 0).mean() 
loss_rate = 1-win_rate

wins = trade_returns[trade_returns > 0]
losses = trade_returns[trade_returns < 0]

avg_win = wins.mean() if len(wins) > 0 else 0
avg_loss = losses.mean() if len(losses) > 0 else 0
profit_factor = wins.sum() / abs(losses.sum()) if abs(losses.sum()) > 0 else np.inf
expectancy = win_rate * avg_win + loss_rate * avg_loss
number_of_trades = len(trade_returns)
exposure = df['Signal'].mean() *100

metrices= {
    'Final Equity' : final_equity,
    'Total Return' : total_return,
    'Sharpe' : sharpe ,
    'Max Drawdown' : max_drawdown,
    'Win Rate % ' : win_rate * 100,
    'Average Win' : avg_win ,
    'Average Loss' : avg_loss,
    'Profit Factor' : profit_factor,
    'Expectancy' : expectancy,
    'Number Of Trades' : number_of_trades,
    'Exposure' : exposure    
}
df1 = pd.DataFrame([metrices])
df1


,Final Equity,Total Return,Sharpe,Max Drawdown,Win Rate %,Average Win,Average Loss,Profit Factor,Expectancy,Number Of Trades,Exposure
0,1.558111,0.558111,0.479912,-0.069565,100.0,0.114286,0,inf,0.114286,1,82.5
